In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
dataset = load_dataset("hatexplain")
print(dataset['train'][0])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load the HateXplain dataset
def load_and_preprocess_dataset():
    # Load the dataset
    dataset = load_dataset("hatexplain")

    # Extract post tokens and convert to text
    texts = [' '.join(post) for post in dataset['train']['post_tokens']]

    # Label extraction strategy
    # We'll consider a sample as hate speech if the majority of annotators label it as hate
    def extract_label(annotators):
        # Count the number of hate labels (assuming 2 indicates hate)
        hate_count = sum(1 for label in annotators['label'] if label == 2)
        total_annotators = len(annotators['label'])

        # If more than half of annotators label it as hate, consider it hate speech
        return 1 if hate_count > total_annotators / 2 else 0

    labels = [extract_label(annotators) for annotators in dataset['train']['annotators']]

    # Split the data into train and validation sets
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.2, random_state=42
    )

    return train_texts, val_texts, train_labels, val_labels

# Tokenization function
def tokenize_function(texts, tokenizer, max_length=512):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

# Custom Dataset class
class HateXplainDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Compute metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calculate precision, recall, f1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    accuracy = accuracy_score(labels, preds)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Main training function
def train_hate_speech_model():
    # Check for GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load texts and labels
    train_texts, val_texts, train_labels, val_labels = load_and_preprocess_dataset()

    # Print some statistics
    print(f"Total training samples: {len(train_texts)}")
    print(f"Hate speech samples: {sum(train_labels)}")
    print(f"Non-hate speech samples: {len(train_labels) - sum(train_labels)}")

    # Choose a pre-trained model (you can experiment with different models)
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2  # Binary classification
    )

    # Tokenize the data
    train_encodings = tokenize_function(train_texts, tokenizer)
    val_encodings = tokenize_function(val_texts, tokenizer)

    # Create datasets
    train_dataset = HateXplainDataset(train_encodings, train_labels)
    val_dataset = HateXplainDataset(val_encodings, val_labels)

    # Training arguments
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        evaluation_strategy="epoch"
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train the model
    trainer.train()

    # Save the model
    model.save_pretrained('./hate_speech_model')
    tokenizer.save_pretrained('./hate_speech_model')

    print("Training complete. Model saved in './hate_speech_model'")

# Run the training
train_hate_speech_model()

was show 18 hours for one epoch!!!

In [ ]:
# import os
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
# from datasets import load_dataset
# from sklearn.model_selection import train_test_split
# import numpy as np
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# # Disable wandb
# os.environ['WANDB_DISABLED'] = 'true'

# # Load the HateXplain dataset
# def load_and_preprocess_dataset():
#     # Load the dataset
#     dataset = load_dataset("hatexplain")

#     # Extract post tokens and convert to text
#     texts = [' '.join(post) for post in dataset['train']['post_tokens']]

#     # Label extraction strategy
#     def extract_label(annotators):
#         hate_count = sum(1 for label in annotators['label'] if label == 2)
#         total_annotators = len(annotators['label'])

#         return 1 if hate_count > total_annotators / 2 else 0

#     labels = [extract_label(annotators) for annotators in dataset['train']['annotators']]

#     # Split the data into train and validation sets
#     train_texts, val_texts, train_labels, val_labels = train_test_split(
#         texts, labels, test_size=0.2, random_state=42
#     )

#     return train_texts, val_texts, train_labels, val_labels

# # Tokenization function
# def tokenize_function(texts, tokenizer, max_length=512):
#     return tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         max_length=max_length,
#         return_tensors="pt"
#     )

# # Custom Dataset class
# class HateXplainDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item['labels'] = torch.tensor(self.labels[idx])
#         return item

#     def __len__(self):
#         return len(self.labels)

# # Compute metrics function
# def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)

#     precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
#     accuracy = accuracy_score(labels, preds)

#     return {
#         'accuracy': accuracy,
#         'precision': precision,
#         'recall': recall,
#         'f1': f1
#     }

# # Main training function
# def train_hate_speech_model():
#     # Check for GPU
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Using device: {device}")

#     # Load texts and labels
#     train_texts, val_texts, train_labels, val_labels = load_and_preprocess_dataset()

#     # Print dataset statistics
#     print(f"Total training samples: {len(train_texts)}")
#     print(f"Hate speech samples: {sum(train_labels)}")
#     print(f"Non-hate speech samples: {len(train_labels) - sum(train_labels)}")

#     # Choose a lighter model - DistilBERT
#     model_name = "distilbert-base-uncased"
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=2  # Binary classification
#     )

#     # Tokenize the data
#     train_encodings = tokenize_function(train_texts, tokenizer)
#     val_encodings = tokenize_function(val_texts, tokenizer)

#     # Create datasets
#     train_dataset = HateXplainDataset(train_encodings, train_labels)
#     val_dataset = HateXplainDataset(val_encodings, val_labels)

#     # Training arguments - reduced batch size for faster training
#     training_args = TrainingArguments(
#         output_dir='./hate_speech_results',
#         num_train_epochs=2,  # Reduced epochs
#         per_device_train_batch_size=8,  # Smaller batch size
#         per_device_eval_batch_size=32,
#         warmup_steps=300,
#         weight_decay=0.01,
#         logging_dir='./logs',
#         logging_steps=10,
#         evaluation_strategy="epoch"
#     )

#     # Initialize Trainer
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#         compute_metrics=compute_metrics
#     )

#     # Train the model
#     trainer.train()

#     # Save the model
#     model.save_pretrained('./hate_speech_model_distilbert')
#     tokenizer.save_pretrained('./hate_speech_model_distilbert')

#     print("Training complete. Model saved in './hate_speech_model_distilbert'")

# # Run the training
# train_hate_speech_model()